# Qwen3.5-9B · P5 · 학습률 3조건

학습 640² / 추론 672² / answer_only(정답+종료 토큰) / **학습·추론 모두 P5**.
고정 학습 ID와 검증 500개, NF4, seed, LoRA, 1 epoch, optimizer·scheduler 등은 baseline과 동일합니다.

| 실행 순서 | 학습률 | 용도 |
|---|---|---|
| 1 | 1e-4 | P5 학습의 새 기준 |
| 2 | 2e-5 | 낮은 학습률 |
| 3 | 5e-5 | 중간 학습률 |

기존 P5 94.2%(471/500)는 P0로 학습하고 P5로 추론한 결과입니다. 이번 LR 비교 기준으로 재사용하지 않습니다.
세 조건 모두 동일한 원본 base 모델에서 새 LoRA를 초기화합니다. 기존 answer_only 어댑터는 학습 출처 검증에만 사용합니다.
학습 데이터량·증강·dev·프롬프트별 추가 비교는 하지 않습니다. test/holdout 추론 및 제출도 없습니다.

### 실행과 시간
1. baseline을 실행했던 프로젝트 폴더에서 열고 `BASELINE_RUN_DIR`, `LOSS_RUN_DIR`, `ENV_PYTHON`을 확인하세요.
2. Run All: 입력 확인 → 1e-4 학습/평가 → 2e-5 → 5e-5 → 최종 표.
3. 각 조건 종료 시 `lr_comparison.csv`가 갱신되고 셀 아래에도 표시됩니다.

**새 학습 3회 + 검증 500개 × 3회(1,500문항)**. 각 학습 저장/재로드 확인 최대 5문항씩 2회, 평가 워밍업 1문항도 포함합니다.
기존 P0/640 answer_only 학습 44분19초와 P5 평가 8분21초를 단순 합산하면 약 2시간38분입니다.
이는 이번 학습을 실측한 값이 아닙니다. P5의 긴 학습 입력과 모델 로딩·검사 비용 때문에 **약 3시간 전후, 더 길어질 수 있습니다**.
원래 384² baseline 전체 실행 완료 시간이 없어 정확한 배수는 미확정입니다. 기준 시간이 B분이면 예상 배수는 약 180/B입니다.
첫 조건이 끝나면 실제 학습·평가 시간을 보고 남은 시간을 갱신하세요.

완료 조건은 해시 확인 후 건너뜁니다. 중단된 학습은 해당 조건을 처음부터 재학습하고 부분 로그를 보존합니다.
학습 완료 후 평가만 중단됐다면 학습은 재사용하고 그 조건의 검증을 처음부터 평가합니다.
OOM은 기록 후 다음 조건으로 이동하며 설정을 자동 변경하지 않습니다. 실행 중 다른 GPU 노트북은 멈추세요.
작성 단계의 CPU 로직·문법 검증과 사용자 GPU 실행은 구분합니다. 02 검토 및 최고 모델 교체는 아직 미완료입니다.


## 1. 기준 실행 폴더 지정
다른 컴퓨터에서 파일을 복사할 필요는 없습니다. 해당 컴퓨터에서 완료한 baseline 출력 폴더를 사용합니다.

In [6]:
from pathlib import Path
import os, sys, json, hashlib, subprocess, time
PROJECT_DIR = Path.cwd().resolve()
BASELINE_RUN_DIR = None  # 예: PROJECT_DIR / "output/TASK-006/TASK006-xxxxxxxxxxxxxxxx"
ENV_PYTHON = PROJECT_DIR / "downloads/envs/TASK006_baseline_qwen35" / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
INFERENCE_RESOLUTION = 672  # 고정 추론 해상도
RESOLUTIONS = ["lr_1e-4", "lr_2e-5", "lr_5e-5"]
LEARNING_RATES = {"lr_1e-4":1e-4, "lr_2e-5":2e-5, "lr_5e-5":5e-5}
LOSS_RUN_DIR = PROJECT_DIR / "output/TASK-006-loss/LOSS-51ca09225d304a9a"
P5_INSTRUCTION = '질문이 요구하는 대상과 조건을 먼저 확인하세요.\n이미지의 큰 글자나 제목을 우선 읽고, 그 내용이 질문 및 네 선지와 연결되는지 대조하세요.\n큰 글자에서 선지에 해당하는 내용을 찾지 못하거나 정답을 결정할 근거가 부족하면,\n작은 글자·설명문·주석·가격·숫자·단위까지 확인 범위를 넓히세요.\n질문이 작은 글자나 특정 위치를 직접 지목하면 해당 부분을 우선 확인하세요.\n큰 글자와 일부 단어가 같다는 이유만으로 선택하지 말고 질문의 조건 전체에 부합하는 선지 하나를 고르세요.\n설명 없이 a, b, c, d 중 소문자 한 글자만 출력하세요.'
SESSION_TAG = "lr_P5_640_672_v1"
RETRY_OOM = False  # 이미 OOM으로 기록된 조건도 재시도하려면 True

if INFERENCE_RESOLUTION is None:
    raise RuntimeError("추론 해상도 미정입니다. 결과 확인 후 INFERENCE_RESOLUTION에 선택값을 입력하세요.")
if not isinstance(INFERENCE_RESOLUTION,int) or isinstance(INFERENCE_RESOLUTION,bool) or INFERENCE_RESOLUTION<=0:
    raise ValueError("INFERENCE_RESOLUTION은 양의 정수여야 합니다.")
if BASELINE_RUN_DIR is None:
    candidates = sorted(p.parent for p in (PROJECT_DIR / "output/TASK-006").glob("TASK006-*/lora_eval_complete.json"))
    if len(candidates) != 1:
        print("baseline 후보:")
        for p in candidates: print(p)
        raise RuntimeError("BASELINE_RUN_DIR에 비교할 baseline 결과 폴더를 지정하세요.")
    BASELINE_RUN_DIR = candidates[0]
BASELINE_RUN_DIR = Path(BASELINE_RUN_DIR).resolve()
for name in ["run_config.json","task006_worker.py","audit_complete.json","train_complete.json","lora_eval_complete.json","requirements.lock.txt","model_assets.json"]:
    if not (BASELINE_RUN_DIR / name).is_file(): raise FileNotFoundError(BASELINE_RUN_DIR / name)
if not ENV_PYTHON.is_file(): raise FileNotFoundError(f"baseline 전용 Python 경로를 확인하세요: {ENV_PYTHON}")
print("baseline:",BASELINE_RUN_DIR)
print("학습 640² / 추론 672², P5 LR 비교:",RESOLUTIONS)


baseline: C:\Users\SSAFY\Desktop\AI2_Challenge\output\TASK-006\TASK006-8f2c71505c11284c
학습 640² / 추론 672², P5 LR 비교: ['lr_1e-4', 'lr_2e-5', 'lr_5e-5']


## 2. 실행 함수 정의
아래 코드는 별도 GPU 프로세스로 실행됩니다. 기존 checkpoint와 분할을 수정하지 않습니다.

In [7]:
RESOLUTION_WORKER = 'import csv, hashlib, importlib.util, json, os, sys, time, uuid\nfrom pathlib import Path\n\n\ndef digest(path):\n    h=hashlib.sha256()\n    with open(path,\'rb\') as f:\n        for b in iter(lambda:f.read(4*1024*1024),b\'\'):h.update(b)\n    return h.hexdigest()\n\n\ndef verify_worker_source(path, expected):\n    data=Path(path).read_bytes()\n    raw=hashlib.sha256(data).hexdigest()\n    lf=hashlib.sha256(data.replace(b"\\r\\n",b"\\n")).hexdigest()\n    if expected not in (raw,lf):\n        raise RuntimeError(f"baseline 코드 내용이 다릅니다. expected={expected}, raw={raw}, LF={lf}. 기존 파일/설정을 수정하지 말고 실행 폴더를 확인하세요.")\n    return expected\n\n\ndef read(path): return json.loads(Path(path).read_text(encoding=\'utf-8\'))\n\n\ndef write(path,obj):\n    p=Path(path);tmp=p.with_name(p.name+\'.tmp\')\n    tmp.write_text(json.dumps(obj,ensure_ascii=False,indent=2,default=str),encoding=\'utf-8\');tmp.replace(p)\n\n\ndef verified_status(root,size):\n    p=root/f\'res_{size}_status.json\'\n    if not p.exists(): return {\'state\':\'not_run\'}\n    rec=read(p)\n    if rec[\'state\']==\'completed\':\n        for name,sha in rec[\'artifacts\'].items():\n            f=Path(rec[\'directory\'])/name\n            if not f.is_file() or digest(f)!=sha: raise RuntimeError(f\'완료 결과 변경/누락: {f}\')\n    return rec\n\n\ndef paired(a,b):\n    m=a.merge(b,on=\'id\',validate=\'one_to_one\',suffixes=(\'_reference\',\'_new\'))\n    if len(m)!=len(a) or len(m)!=len(b) or not (m.gold_reference==m.gold_new).all():\n        raise RuntimeError(\'비교 ID/정답이 동일하지 않습니다.\')\n    if not (m.group_id_reference==m.group_id_new).all(): raise RuntimeError(\'이미지 그룹 불일치\')\n    old=m.answer_reference==m.gold_reference;new=m.answer_new==m.gold_new\n    m[\'transition\']=[\'gain\' if y and not x else \'loss\' if x and not y else \'same_correct\' if x else \'same_wrong\' for x,y in zip(old,new)]\n    return m,{\'gain\':int((~old & new).sum()),\'loss\':int((old & ~new).sum()),\n              \'delta_pp\':100*float(new.mean()-old.mean())}\n\n\ndef summarize(cfg):\n    import pandas as pd\n    root=Path(cfg[\'output_dir\']);rows=[];preds={};types=[]\n    for name in cfg[\'resolutions\']:\n        rec=verified_status(root,name)\n        r={\'condition\':name,\'learning_rate\':cfg[\'learning_rates\'][name],\'status\':rec[\'state\'],\n           \'loss_mode\':\'answer_only\',\'prompt\':\'P5\',\'train_resolution\':640,\'inference_resolution\':672}\n        tr=root/f\'train_{name}_status.json\'\n        if tr.exists():\n            t=read(tr);r[\'train_status\']=t[\'state\']\n            if t[\'state\']==\'completed\':r[\'training_seconds\']=t[\'training\'][\'seconds\']\n        if rec[\'state\']==\'completed\':\n            d=Path(rec[\'directory\']);r.update(read(d/\'valid_metrics.json\'));r[\'accuracy_pct\']=100*r[\'accuracy\']\n            r[\'parse_failure_pct\']=100*r[\'parse_failure_rate\']\n            preds[name]=pd.read_csv(d/\'valid_predictions.csv\',keep_default_na=False)\n        else:r[\'error\']=rec.get(\'error\',\'\')\n        rows.append(r)\n    for r in rows:\n        name=r[\'condition\']\n        if name not in preds:continue\n        if \'lr_1e-4\' in preds:\n            m,stats=paired(preds[\'lr_1e-4\'],preds[name]);r.update(stats)\n            m.to_csv(root/f\'paired_1e-4_vs_{name}.csv\',index=False,encoding=\'utf-8-sig\')\n            m[m.transition.isin([\'gain\',\'loss\'])].to_csv(root/f\'changed_1e-4_vs_{name}.csv\',index=False,encoding=\'utf-8-sig\')\n        for kind,g in preds[name].groupby(\'question_type\'):\n            types.append({\'condition\':name,\'question_type\':kind,\'n\':len(g),\'accuracy\':float((g.answer==g.gold).mean())})\n    pd.DataFrame(types).to_csv(root/\'type_comparison.csv\',index=False,encoding=\'utf-8-sig\')\n    table=pd.DataFrame(rows);table.to_csv(root/\'lr_comparison.csv\',index=False,encoding=\'utf-8-sig\')\n    write(root/\'summary.json\',{\'results\':rows,\'adoption\':\'pending\',\'review_02\':\'pending\',\'test_used\':False,\'holdout_evaluated\':False})\n    (root/\'PROJECT_STATUS_update.md\').write_text(\'# TASK-006 LR / P5\\n\\n\'+table.to_string(index=False)+\'\\n\\n채택·독립 검토 미완료. P5 학습을 새로 수행한 LR 비교.\',encoding=\'utf-8\')\n    print(table.to_string(index=False),flush=True)\n    return rows\n\n\ndef load_baseline(cfg):\n    base=Path(cfg[\'baseline_dir\']);original=read(base/\'run_config.json\')\n    worker=base/\'task006_worker.py\'\n    if verify_worker_source(worker,original[\'worker_sha256\'])!=cfg[\'baseline_worker_sha256\']:\n        raise RuntimeError(\'baseline 실행 코드 해시 불일치\')\n    spec=importlib.util.spec_from_file_location(\'saved_baseline\',worker)\n    module=importlib.util.module_from_spec(spec);sys.modules[spec.name]=module;spec.loader.exec_module(module)\n    module.block_network()\n    def p5_prompt(question,a,b,c,d):\n        return f\'{question}\\n(a) {a}\\n(b) {b}\\n(c) {c}\\n(d) {d}\\n\\n\'+cfg[\'p5_instruction\']\n    module.build_mc_prompt=p5_prompt\n    return module,original\n\n\ndef prepare(cfg):\n    import pandas as pd\n    root=Path(cfg[\'output_dir\']);base=Path(cfg[\'baseline_dir\']);module,original=load_baseline(cfg)\n    train=module.completed(base,\'train\');evaluation=module.completed(base,\'lora_eval\')\n    audit=module.completed(base,\'audit\')\n    if train is None or evaluation is None or audit is None:raise RuntimeError(\'baseline audit/train/lora_eval을 먼저 완료하세요.\')\n    if original[\'pixel_budget\']!=384**2:raise RuntimeError(\'학습 해상도 384²인 baseline을 지정하세요.\')\n    reload_check=read(Path(train[\'directory\'])/\'reload_check.json\')\n    if not reload_check.get(\'identical_outputs\'):raise RuntimeError(\'baseline 저장/재로드 검증 미통과\')\n    fixed_train,valid,info=module.load_split(original,base)\n    if len(valid)!=500:raise RuntimeError(f\'고정 검증 500개가 아닙니다: {len(valid)}. 기존 분할 확인 필요\')\n    if original[\'epochs\']!=1:raise RuntimeError(\'이번 코드는 1 epoch 비교입니다.\')\n    loss_dir=Path(cfg[\'loss_dir\']);record_path=loss_dir/\'train_answer_only_status.json\'\n    if digest(record_path)!=cfg[\'loss_record_sha256\']:raise RuntimeError(\'선택 loss 기록 변경\')\n    rec=read(record_path)\n    if rec[\'state\']!=\'completed\':raise RuntimeError(\'answer_only 학습 미완료\')\n    for name,h in rec[\'artifacts\'].items():\n        if digest(Path(rec[\'directory\'])/name)!=h:raise RuntimeError(\'answer_only 학습 artifact 변경\')\n    prior=read(Path(rec[\'directory\'])/\'training_config.json\')\n    for key in original:\n        if key not in [\'run_dir\',\'pixel_budget\',\'image_policy\',\'loss\',\'loss_mode\'] and prior.get(key)!=original[key]:\n            raise RuntimeError(f\'선택 학습과 baseline 설정 차이: {key}\')\n    if prior[\'pixel_budget\']!=640**2 or prior[\'loss_mode\']!=\'answer_only\':raise RuntimeError(\'선택 loss 조건 불일치\')\n    lf=read(loss_dir/\'frozen_inputs.json\')\n    if lf[\'data_manifest\']!=info or lf[\'baseline_config_sha256\']!=digest(base/\'run_config.json\'):raise RuntimeError(\'선택 loss의 데이터/기준 설정 불일치\')\n    if not read(Path(rec[\'directory\'])/\'reload_check.json\').get(\'identical_outputs\'):raise RuntimeError(\'선택 loss 재로드 미검증\')\n    fixed_train[[\'id\',\'group_id\',\'question_type\']].to_csv(root/\'fixed_train_ids.csv\',index=False)\n    write(root/\'prompt.json\',{\'name\':\'P5\',\'instruction\':cfg[\'p5_instruction\'],\'system\':module.SYSTEM_INSTRUCT,\n         \'example\':module.build_mc_prompt(*(fixed_train.iloc[0][k] for k in [\'question\',\'a\',\'b\',\'c\',\'d\']))})\n    print(\'고정 학습:\',len(fixed_train),\'검증:\',len(valid),\'새 학습 3회, P5 train + inference\',flush=True)\n    # Avoid relying only on paths; check every model asset against baseline download hashes.\n    assets=read(base/\'model_assets.json\')\n    if assets[\'revision\']!=original[\'revision\']:raise RuntimeError(\'모델 revision 불일치\')\n    for name,record in assets[\'files\'].items():\n        f=Path(original[\'model_dir\'])/name\n        if not f.is_file() or digest(f)!=record[\'sha256\']:raise RuntimeError(f\'모델 파일 확인 필요: {name}\')\n    valid[[\'id\',\'group_id\',\'question_type\']].to_csv(root/\'fixed_valid_ids.csv\',index=False)\n    frozen={\'baseline_config\':original,\'data_manifest\':info,\'checkpoint\':str(Path(train[\'directory\'])/\'adapter_epoch1\'),\n            \'baseline_records\':{name:digest(base/(name+\'_complete.json\')) for name in [\'audit\',\'train\',\'lora_eval\']},\n            \'baseline_config_sha256\':digest(base/\'run_config.json\'),\'valid_ids_sha256\':digest(root/\'fixed_valid_ids.csv\'),\n            \'train_ids_sha256\':digest(root/\'fixed_train_ids.csv\'),\'train_n\':len(fixed_train),\'valid_n\':len(valid),\'previous_predictions\':str(Path(evaluation[\'directory\'])/\'valid_lora_predictions.csv\'),\n            \'model_file_stats\':{name:[(Path(original[\'model_dir\'])/name).stat().st_size,(Path(original[\'model_dir\'])/name).stat().st_mtime_ns] for name in assets[\'files\']}}\n    frozen_path=root/\'frozen_inputs.json\'\n    if frozen_path.exists() and read(frozen_path)!=frozen:raise RuntimeError(\'기준 입력 변경. 새 SESSION_TAG로 실행하세요.\')\n    write(frozen_path,frozen)\n    print(\'준비 완료. 이력 LoRA (학습에 재사용하지 않음):\',frozen[\'checkpoint\'],\'검증:\',len(valid),flush=True)\n    summarize(cfg)\n\n\ndef configure_pixels(adapter,pixels):\n    ip=adapter.processor.image_processor\n    ip.size={\'shortest_edge\':pixels,\'longest_edge\':pixels}\n    if hasattr(ip,\'min_pixels\'):ip.min_pixels=pixels\n    if hasattr(ip,\'max_pixels\'):ip.max_pixels=pixels\n\n\ndef validate_inputs(cfg):\n    import pandas as pd\n    root=Path(cfg[\'output_dir\']);base=Path(cfg[\'baseline_dir\']);frozen=read(root/\'frozen_inputs.json\')\n    module,original=load_baseline(cfg)\n    if digest(base/\'run_config.json\')!=frozen[\'baseline_config_sha256\']:raise RuntimeError(\'baseline 설정 변경\')\n    for name,sha in frozen[\'baseline_records\'].items():\n        if digest(base/(name+\'_complete.json\'))!=sha:raise RuntimeError(\'baseline 완료 기록 변경\')\n    module.completed(base,\'train\')\n    train,valid,info=module.load_split(original,base)\n    if digest(root/\'fixed_train_ids.csv\')!=frozen[\'train_ids_sha256\'] or pd.read_csv(root/\'fixed_train_ids.csv\',dtype=str).id.tolist()!=train.id.tolist():raise RuntimeError(\'학습 ID/순서 변경\')\n    if info!=frozen[\'data_manifest\']:raise RuntimeError(\'분할 manifest 변경\')\n    if digest(root/\'fixed_valid_ids.csv\')!=frozen[\'valid_ids_sha256\']:raise RuntimeError(\'검증 ID 파일 변경\')\n    if pd.read_csv(root/\'fixed_valid_ids.csv\',dtype=str).id.tolist()!=valid.id.tolist():raise RuntimeError(\'검증 순서 변경\')\n    for name,stat in frozen[\'model_file_stats\'].items():\n        f=Path(original[\'model_dir\'])/name\n        if [f.stat().st_size,f.stat().st_mtime_ns]!=stat:raise RuntimeError(\'모델 파일 변경\')\n    return module,original,frozen,train,valid\n\n\ndef answer_span(full,empty,eos):\n    start=0\n    while start<min(len(full),len(empty)) and full[start]==empty[start]:start+=1\n    suffix=0\n    while suffix<min(len(full)-start,len(empty)-start) and full[-1-suffix]==empty[-1-suffix]:suffix+=1\n    end=len(full)-suffix\n    if start>=end:raise RuntimeError(\'정답 token span을 찾지 못했습니다.\')\n    if end>=len(full) or full[end]!=eos:\n        raise RuntimeError(\'정답 바로 뒤 종료 토큰이 예상과 다릅니다. 템플릿 검토 필요; 자동 우회 없음.\')\n    return start,end\n\n\ndef install_answer_collator(module):\n    import copy\n    original=module.DataCollator\n    class AnswerOnlyCollator(original):\n        def __call__(self,batch):\n            enc=super().__call__(batch)\n            if not self.train:return enc\n            if len(batch)!=1:raise ValueError(\'이 실험은 baseline과 동일하게 batch_size=1\')\n            empty=[]\n            for sample in batch:\n                msgs=copy.deepcopy(sample[\'messages\'])\n                if msgs[-1][\'role\']!=\'assistant\':raise RuntimeError(\'assistant 정답 없음\')\n                gold=msgs[-1][\'content\'][0][\'text\']\n                msgs[-1][\'content\']=[{\'type\':\'text\',\'text\':\'\'}]\n                empty.append({\'messages\':msgs,\'image\':sample[\'image\']})\n            empty_enc=super().__call__(empty)\n            full=enc[\'input_ids\'][0].tolist();blank=empty_enc[\'input_ids\'][0].tolist()\n            start,end=answer_span(full,blank,self.processor.tokenizer.eos_token_id)\n            decoded=self.processor.tokenizer.decode(full[start:end],skip_special_tokens=False).strip()\n            if decoded!=gold:raise RuntimeError(f\'정답 토큰 검증 실패: {decoded!r} != {gold!r}\')\n            enc[\'labels\'].fill_(-100)\n            enc[\'labels\'][0,start:end+1]=enc[\'input_ids\'][0,start:end+1]\n            return enc\n    module.DataCollator=AnswerOnlyCollator\n\n\ndef audit_targets(module,adapter,train,out):\n    import pandas as pd\n    rows=[]\n    for row in train.head(3).to_dict(\'records\'):\n        enc=adapter.encode(row,training=True)\n        ids=enc[\'input_ids\'][0].tolist();labels=enc[\'labels\'][0].tolist()\n        for i,t in enumerate(ids):\n            rows.append({\'id\':row[\'id\'],\'position\':i,\'token_id\':t,\n                         \'token\':adapter.tokenizer.convert_ids_to_tokens(t),\'supervised\':labels[i]!=-100})\n        del enc\n    pd.DataFrame(rows).to_csv(out/\'supervised_tokens.csv\',index=False,encoding=\'utf-8-sig\')\n\n\ndef train_condition(cfg,size):\n    import torch\n    root=Path(cfg[\'output_dir\']);module,original,frozen,train,valid=validate_inputs(cfg)\n    status=root/f\'train_{size}_status.json\'\n    if status.exists():\n        old=read(status)\n        if old[\'state\']==\'completed\':\n            for name,h in old.get(\'artifacts\',{}).items():\n                if digest(Path(old[\'directory\'])/name)!=h:raise RuntimeError(\'학습 결과 파일 변경\')\n            print(\'완료 학습 재사용:\',size,flush=True);return\n        if old[\'state\']==\'blocked_oom\' and not cfg[\'retry_oom\']:return\n    out=root/f\'train_{size}_{time.strftime("%Y%m%d_%H%M%S")}_{uuid.uuid4().hex[:8]}\';out.mkdir()\n    settings=dict(original);settings[\'pixel_budget\']=640**2;settings[\'run_dir\']=str(out)\n    settings[\'image_policy\']=\'train pixel budget 640 squared\'\n    settings[\'loss_mode\']=\'answer_only\';settings[\'loss\']=\'answer_and_eos_only\'\n    settings[\'learning_rate\']=cfg[\'learning_rates\'][size];settings[\'prompt\']=\'P5\';settings[\'p5_instruction\']=cfg[\'p5_instruction\']\n    write(out/\'training_config.json\',settings);write(status,{\'state\':\'running\',\'directory\':str(out)})\n    try:\n        module.gpu_environment(settings,out)\n        install_answer_collator(module)\n        adapter=module.ModelAdapter(settings,out)\n        adapter.add_lora()  # Fresh base and fresh LoRA; no preceding resolution adapter loaded.\n        audit_targets(module,adapter,train,out)\n        training=module.train_one_epoch(adapter,train,settings,out)\n        checkpoint=out/\'adapter_epoch1\'\n        adapter.model.save_pretrained(checkpoint);adapter.processor.save_pretrained(checkpoint)\n        # Use fixed inference budget for both sides of serialization check.\n        configure_pixels(adapter,cfg[\'inference_resolution\']**2)\n        probe=valid.head(min(5,len(valid)))\n        before,_=module.evaluate(adapter,probe,\'reload_before\',out)\n        adapter.reload_adapter(checkpoint)\n        after,_=module.evaluate(adapter,probe,\'reload_after\',out)\n        keys=[\'id\',\'answer\',\'raw_output\']\n        identical=all(all(x[k]==y[k] for k in keys) for x,y in zip(before,after)) and len(before)==len(after)\n        write(out/\'reload_check.json\',{\'n\':len(probe),\'identical_outputs\':identical})\n        if not identical:raise RuntimeError(\'저장/재로드 예측 불일치. 결과 검토 필요\')\n        artifacts={str(p.relative_to(out)):digest(p) for p in out.rglob(\'*\') if p.is_file()}\n        write(status,{\'state\':\'completed\',\'directory\':str(out),\'checkpoint\':str(checkpoint),\n                      \'training\':training,\'artifacts\':artifacts,\'reused_baseline\':False})\n    except torch.cuda.OutOfMemoryError as exc:\n        write(status,{\'state\':\'blocked_oom\',\'directory\':str(out),\'error\':str(exc)})\n    except BaseException as exc:\n        write(status,{\'state\':\'failed_or_interrupted\',\'directory\':str(out),\'error\':str(exc)});raise\n\n\ndef run_resolution(cfg,size):\n    import torch\n    from peft import PeftModel\n    root=Path(cfg[\'output_dir\']);module,original,frozen,train,valid=validate_inputs(cfg)\n    old=verified_status(root,size)\n    if old[\'state\']==\'completed\' or (old[\'state\']==\'blocked_oom\' and not cfg[\'retry_oom\']):\n        summarize(cfg);return\n    trained=read(root/f\'train_{size}_status.json\')\n    status=root/f\'res_{size}_status.json\'\n    if trained[\'state\']!=\'completed\':\n        write(status,{\'state\':trained[\'state\'],\'error\':trained.get(\'error\',\'training incomplete\')});summarize(cfg);return\n    for name,h in trained.get(\'artifacts\',{}).items():\n        if digest(Path(trained[\'directory\'])/name)!=h:raise RuntimeError(\'학습 artifact 변경\')\n    out=root/f\'eval_train{size}_{time.strftime("%Y%m%d_%H%M%S")}_{uuid.uuid4().hex[:8]}\';out.mkdir()\n    settings=dict(original);settings[\'pixel_budget\']=cfg[\'inference_resolution\']**2;settings[\'run_dir\']=str(out)\n    settings.update(loss_mode=\'answer_only\',learning_rate=cfg[\'learning_rates\'][size],prompt=\'P5\',p5_instruction=cfg[\'p5_instruction\'])\n    write(out/\'inference_config.json\',settings);write(status,{\'state\':\'running\',\'directory\':str(out)})\n    try:\n        module.gpu_environment(settings,out);adapter=module.ModelAdapter(settings,out)\n        adapter.model=PeftModel.from_pretrained(adapter.model,trained[\'checkpoint\'],local_files_only=True,is_trainable=False)\n        probe=valid.iloc[0].to_dict();probe.pop(\'answer\',None)\n        adapter.generate(adapter.encode(probe,training=False));torch.cuda.synchronize()\n        _,metrics=module.evaluate(adapter,valid,\'valid\',out)\n        metrics.update({\'train_resolution\':640,\'loss_mode\':\'answer_only\',\'learning_rate\':cfg[\'learning_rates\'][size],\'prompt\':\'P5\',\'inference_resolution\':cfg[\'inference_resolution\'],\n            \'checkpoint\':trained[\'checkpoint\'],\'training_seconds\':trained[\'training\'][\'seconds\'],\n            \'optimizer_updates\':trained[\'training\'][\'updates\'],\n            \'training_peak_allocated_gib\':trained[\'training\'][\'peak_allocated_gib\'],\n            \'reused_baseline\':trained[\'reused_baseline\'],\'warmup_samples\':1})\n        write(out/\'valid_metrics.json\',metrics)\n        artifacts={str(p.relative_to(out)):digest(p) for p in out.rglob(\'*\') if p.is_file()}\n        write(status,{\'state\':\'completed\',\'directory\':str(out),\'artifacts\':artifacts})\n    except torch.cuda.OutOfMemoryError as exc:\n        write(status,{\'state\':\'blocked_oom\',\'directory\':str(out),\'error\':str(exc)})\n    except BaseException as exc:\n        write(status,{\'state\':\'failed_or_interrupted\',\'directory\':str(out),\'error\':str(exc)});raise\n    finally:\n        with open(root/\'CHANGELOG.md\',\'a\',encoding=\'utf-8\') as f:f.write(f"\\n- train {size}, inference {cfg[\'inference_resolution\']}: {read(status)[\'state\']}\\n")\n    summarize(cfg)\n\n\nif __name__==\'__main__\':\n    config=read(sys.argv[1]);stage=sys.argv[2]\n    if stage==\'prepare\':prepare(config)\n    elif stage==\'summary\':summarize(config)\n    else:\n        is_train=stage.startswith(\'train_\')\n        size=stage.removeprefix(\'train_\')\n        if size not in config[\'resolutions\']:raise ValueError(\'설정에 없는 해상도\')\n        if is_train:train_condition(config,size)\n        else:run_resolution(config,size)\n'

In [8]:
def file_hash(p):
    h=hashlib.sha256()
    with open(p,"rb") as f:
        for b in iter(lambda:f.read(4*1024*1024),b""): h.update(b)
    return h.hexdigest()
base_config=json.loads((BASELINE_RUN_DIR/"run_config.json").read_text(encoding="utf-8"))
if base_config.get("model_id")!="Qwen/Qwen3.5-9B":raise RuntimeError("Qwen3.5-9B baseline을 지정하세요.")
worker_bytes=(BASELINE_RUN_DIR/"task006_worker.py").read_bytes()
worker_raw_hash=hashlib.sha256(worker_bytes).hexdigest()
worker_lf_hash=hashlib.sha256(worker_bytes.replace(bytes([13,10]),bytes([10]))).hexdigest()
if base_config["worker_sha256"] not in (worker_raw_hash,worker_lf_hash):
    raise RuntimeError(f"baseline 코드 내용이 다릅니다. expected={base_config['worker_sha256']}, raw={worker_raw_hash}, LF={worker_lf_hash}. 기존 파일/설정을 수정하지 말고 실행 폴더를 확인하세요.")
if worker_raw_hash!=base_config["worker_sha256"]:
    print("Windows CRLF 줄바꿈 차이만 확인됨. LF 정규화 해시 검증 통과.")
# Detect package changes instead of silently comparing different environments.
freeze=subprocess.check_output([str(ENV_PYTHON),"-m","pip","freeze"],text=True,encoding="utf-8")
old=(BASELINE_RUN_DIR/"requirements.lock.txt").read_text(encoding="utf-8")
if sorted(freeze.splitlines())!=sorted(old.splitlines()):
    raise RuntimeError("baseline 실행 후 패키지 구성이 바뀌었습니다. baseline 전용 환경을 복원한 뒤 실행하세요.")
CFG={"baseline_dir":str(BASELINE_RUN_DIR),"loss_dir":str(LOSS_RUN_DIR),"loss_record_sha256":file_hash(LOSS_RUN_DIR/"train_answer_only_status.json"),"learning_rates":LEARNING_RATES,"p5_instruction":P5_INSTRUCTION,"resolutions":RESOLUTIONS,"inference_resolution":INFERENCE_RESOLUTION,"session_tag":SESSION_TAG,
     "baseline_worker_sha256":base_config["worker_sha256"],
     "baseline_config_sha256":file_hash(BASELINE_RUN_DIR/"run_config.json"),
     "train_record_sha256":file_hash(BASELINE_RUN_DIR/"train_complete.json"),
     "audit_record_sha256":file_hash(BASELINE_RUN_DIR/"audit_complete.json"),
     "worker_sha256":hashlib.sha256(RESOLUTION_WORKER.encode()).hexdigest(),
     "environment_sha256":hashlib.sha256(freeze.encode()).hexdigest(),"code_version":"lr-P5-640-672-v1"}
fingerprint=hashlib.sha256(json.dumps(CFG,sort_keys=True).encode()).hexdigest()[:16]
OUTPUT_DIR=PROJECT_DIR/"output/TASK-006-lr"/("LR-"+fingerprint)
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
CFG.update(output_dir=str(OUTPUT_DIR),retry_oom=RETRY_OOM)
WORKER=OUTPUT_DIR/"resolution_worker.py";CONFIG=OUTPUT_DIR/"resolution_config.json"
compile(RESOLUTION_WORKER,str(WORKER),"exec")
WORKER.write_bytes(RESOLUTION_WORKER.encode("utf-8"))
CONFIG.write_text(json.dumps(CFG,ensure_ascii=False,indent=2),encoding="utf-8")
(OUTPUT_DIR/"requirements.lock.txt").write_text(freeze,encoding="utf-8")

def show_table():
    import csv
    from IPython.display import display,Markdown,FileLink
    p=OUTPUT_DIR/"lr_comparison.csv"
    if not p.exists(): return
    with open(p,encoding="utf-8-sig",newline="") as f: rows=list(csv.DictReader(f))
    columns=[("learning_rate","학습률"),("prompt","프롬프트"),("train_resolution","학습 해상도"),("inference_resolution","추론 해상도"),("training_seconds","학습 초"),("status","상태"),("accuracy_pct","Accuracy(%)"),("correct_n","정답 수"),
             ("n","검증 수"),("delta_pp","1e-4 대비 %p"),("gain","개선"),("loss","악화"),
             ("parse_failure_pct","파싱 실패(%)"),("seconds_per_sample","초/문항"),("peak_allocated_gib","최대 VRAM(GiB)")]
    def fmt(value):
        if value in (None,""): return "—"
        try: return f"{float(value):.6g}" if any(c in str(value) for c in '.eE') else str(value)
        except ValueError: return str(value).replace('|','/')
    text='| '+' | '.join(b for a,b in columns)+' |\n| '+' | '.join('---' for _ in columns)+' |\n'
    for row in rows: text+='| '+' | '.join(fmt(row.get(a,'')) for a,b in columns)+' |\n'
    display(Markdown(text));print("결과 파일:",p);display(FileLink(str(p)))

def run_stage(stage):
    for folder in ['TASK-006-loss','TASK-006-prompts','TASK-006-resolution','TASK-006-training-resolution']:
        other=PROJECT_DIR/'output'/folder/'gpu_experiment.lock'
        if other.exists():raise RuntimeError(f'다른 실험 실행 또는 잔여 잠금: {other}')
    # Same resolution project lock prevents simultaneous notebook runs.
    lock=PROJECT_DIR/"output/TASK-006-lr/gpu_experiment.lock"
    if (BASELINE_RUN_DIR/"running.lock").exists():raise RuntimeError("baseline 작업이 실행 중입니다. 종료 후 실행하세요.")
    try: fd=os.open(lock,os.O_CREAT|os.O_EXCL|os.O_WRONLY)
    except FileExistsError:raise RuntimeError(f"다른 해상도 실험이 실행 중이거나 잠금이 남아 있습니다: {lock}. 실행 프로세스가 없을 때만 잠금을 삭제하세요.")
    process=None
    try:
        with os.fdopen(fd,"w") as f:f.write(str(os.getpid()))
        env=os.environ.copy();env.update(PYTHONIOENCODING="utf-8",PYTHONUNBUFFERED="1")
        with open(OUTPUT_DIR/(str(stage)+".log"),"a",encoding="utf-8") as log:
            process=subprocess.Popen([str(ENV_PYTHON),"-u",str(WORKER),str(CONFIG),str(stage)],
                stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,encoding="utf-8",errors="replace",env=env)
            for line in process.stdout:print(line,end="");log.write(line);log.flush()
            if process.wait()!=0:raise RuntimeError(f"단계 {stage} 실패. {OUTPUT_DIR / (str(stage)+'.log')} 확인")
    except BaseException:
        if process is not None and process.poll() is None:
            process.terminate()
            try:process.wait(timeout=10)
            except subprocess.TimeoutExpired:process.kill();process.wait()
        if str(stage) in RESOLUTIONS or str(stage).startswith("train_"):
            status=OUTPUT_DIR/((str(stage) if str(stage).startswith("train_") else "res_"+str(stage))+"_status.json")
            if status.exists():
                rec=json.loads(status.read_text(encoding="utf-8"))
                if rec["state"]=="running":
                    rec["state"]="interrupted";status.write_text(json.dumps(rec,ensure_ascii=False,indent=2),encoding="utf-8")
        raise
    finally:lock.unlink(missing_ok=True)
    show_table()
print("새 실험 결과 폴더:",OUTPUT_DIR)


Windows CRLF 줄바꿈 차이만 확인됨. LF 정규화 해시 검증 통과.
새 실험 결과 폴더: C:\Users\SSAFY\Desktop\AI2_Challenge\output\TASK-006-lr\LR-6007c4ee02657be3


## 3. 기준 입력 확인
checkpoint·코드·패키지·모델 파일·분할·이미지 해시를 확인합니다. 모델 파일 전체를 확인하므로 이 단계는 시간이 걸릴 수 있습니다. 최종 검증은 평가하지 않습니다.

In [9]:
run_stage("prepare")

고정 학습: 1000 검증: 500 새 학습 3회, P5 train + inference
준비 완료. 이력 LoRA (학습에 재사용하지 않음): C:\Users\SSAFY\Desktop\AI2_Challenge\output\TASK-006\TASK006-8f2c71505c11284c\train_20260922_115654_f6f6586f\adapter_epoch1 검증: 500
condition  learning_rate  status   loss_mode prompt  train_resolution  inference_resolution train_status error
  lr_1e-4        0.00010 not_run answer_only     P5               640                   672      running      
  lr_2e-5        0.00002 not_run answer_only     P5               640                   672          NaN      
  lr_5e-5        0.00005 not_run answer_only     P5               640                   672          NaN      


| 학습률 | 프롬프트 | 학습 해상도 | 추론 해상도 | 학습 초 | 상태 | Accuracy(%) | 정답 수 | 검증 수 | 1e-4 대비 %p | 개선 | 악화 | 파싱 실패(%) | 초/문항 | 최대 VRAM(GiB) |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 0.0001 | P5 | 640 | 672 | — | not_run | — | — | — | — | — | — | — | — | — |
| 2e-05 | P5 | 640 | 672 | — | not_run | — | — | — | — | — | — | — | — | — |
| 5e-05 | P5 | 640 | 672 | — | not_run | — | — | — | — | — | — | — | — | — |


결과 파일: C:\Users\SSAFY\Desktop\AI2_Challenge\output\TASK-006-lr\LR-6007c4ee02657be3\lr_comparison.csv


C:\Users\SSAFY\Desktop\AI2_Challenge\output\TASK-006-lr\LR-6007c4ee02657be3\lr_comparison.csv

## 4. P5 새 기준 — LR 1e-4

In [10]:
run_stage("train_" + str(RESOLUTIONS[0]))
run_stage(RESOLUTIONS[0])

{'python': '3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]', 'os': 'Windows-10-10.0.26200-SP0', 'torch': '2.11.0+cu128', 'cuda': '12.8', 'gpu': 'NVIDIA GeForce RTX 5060 Ti', 'vram_gib': 15.92828369140625, 'packages': {'transformers': '5.8.0', 'peft': '0.18.1', 'bitsandbytes': '0.49.2', 'accelerate': '1.12.0'}, 'network': 'blocked; local model files only'}
W0922 18:07:47.928000 9176 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d

Loading weights:   0%|          | 2/760 [00:01<08:55,  1.41it/s]C:\Users\SSAFY\Desktop\AI2_Challenge\downloads\envs\TASK006_baseline_qwen35\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: Fu

| 학습률 | 프롬프트 | 학습 해상도 | 추론 해상도 | 학습 초 | 상태 | Accuracy(%) | 정답 수 | 검증 수 | 1e-4 대비 %p | 개선 | 악화 | 파싱 실패(%) | 초/문항 | 최대 VRAM(GiB) |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 0.0001 | P5 | 640 | 672 | — | not_run | — | — | — | — | — | — | — | — | — |
| 2e-05 | P5 | 640 | 672 | — | not_run | — | — | — | — | — | — | — | — | — |
| 5e-05 | P5 | 640 | 672 | — | not_run | — | — | — | — | — | — | — | — | — |


결과 파일: C:\Users\SSAFY\Desktop\AI2_Challenge\output\TASK-006-lr\LR-6007c4ee02657be3\lr_comparison.csv


C:\Users\SSAFY\Desktop\AI2_Challenge\output\TASK-006-lr\LR-6007c4ee02657be3\lr_comparison.csv

{'python': '3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]', 'os': 'Windows-10-10.0.26200-SP0', 'torch': '2.11.0+cu128', 'cuda': '12.8', 'gpu': 'NVIDIA GeForce RTX 5060 Ti', 'vram_gib': 15.92828369140625, 'packages': {'transformers': '5.8.0', 'peft': '0.18.1', 'bitsandbytes': '0.49.2', 'accelerate': '1.12.0'}, 'network': 'blocked; local model files only'}
W0922 18:53:54.533000 19320 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d

Loading weights:   0%|          | 2/760 [00:01<09:07,  1.38it/s]C:\Users\SSAFY\Desktop\AI2_Challenge\downloads\envs\TASK006_baseline_qwen35\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: F

| 학습률 | 프롬프트 | 학습 해상도 | 추론 해상도 | 학습 초 | 상태 | Accuracy(%) | 정답 수 | 검증 수 | 1e-4 대비 %p | 개선 | 악화 | 파싱 실패(%) | 초/문항 | 최대 VRAM(GiB) |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 0.0001 | P5 | 640 | 672 | 2729.79 | completed | 94 | 470 | 500 | 0 | 0 | 0 | 0 | 0.865942 | 7.68746 |
| 2e-05 | P5 | 640 | 672 | — | not_run | — | — | — | — | — | — | — | — | — |
| 5e-05 | P5 | 640 | 672 | — | not_run | — | — | — | — | — | — | — | — | — |


결과 파일: C:\Users\SSAFY\Desktop\AI2_Challenge\output\TASK-006-lr\LR-6007c4ee02657be3\lr_comparison.csv


C:\Users\SSAFY\Desktop\AI2_Challenge\output\TASK-006-lr\LR-6007c4ee02657be3\lr_comparison.csv

## 5. LR 2e-5

In [11]:
run_stage("train_" + str(RESOLUTIONS[1]))
run_stage(RESOLUTIONS[1])

{'python': '3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]', 'os': 'Windows-10-10.0.26200-SP0', 'torch': '2.11.0+cu128', 'cuda': '12.8', 'gpu': 'NVIDIA GeForce RTX 5060 Ti', 'vram_gib': 15.92828369140625, 'packages': {'transformers': '5.8.0', 'peft': '0.18.1', 'bitsandbytes': '0.49.2', 'accelerate': '1.12.0'}, 'network': 'blocked; local model files only'}
W0922 19:01:35.337000 14216 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d

Loading weights:   0%|          | 2/760 [00:01<08:51,  1.43it/s]C:\Users\SSAFY\Desktop\AI2_Challenge\downloads\envs\TASK006_baseline_qwen35\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: F

| 학습률 | 프롬프트 | 학습 해상도 | 추론 해상도 | 학습 초 | 상태 | Accuracy(%) | 정답 수 | 검증 수 | 1e-4 대비 %p | 개선 | 악화 | 파싱 실패(%) | 초/문항 | 최대 VRAM(GiB) |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 0.0001 | P5 | 640 | 672 | 2729.79 | completed | 94 | 470 | 500 | 0 | 0 | 0 | 0 | 0.865942 | 7.68746 |
| 2e-05 | P5 | 640 | 672 | — | not_run | — | — | — | — | — | — | — | — | — |
| 5e-05 | P5 | 640 | 672 | — | not_run | — | — | — | — | — | — | — | — | — |


결과 파일: C:\Users\SSAFY\Desktop\AI2_Challenge\output\TASK-006-lr\LR-6007c4ee02657be3\lr_comparison.csv


C:\Users\SSAFY\Desktop\AI2_Challenge\output\TASK-006-lr\LR-6007c4ee02657be3\lr_comparison.csv

{'python': '3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]', 'os': 'Windows-10-10.0.26200-SP0', 'torch': '2.11.0+cu128', 'cuda': '12.8', 'gpu': 'NVIDIA GeForce RTX 5060 Ti', 'vram_gib': 15.92828369140625, 'packages': {'transformers': '5.8.0', 'peft': '0.18.1', 'bitsandbytes': '0.49.2', 'accelerate': '1.12.0'}, 'network': 'blocked; local model files only'}
W0922 19:48:08.100000 4612 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d

Loading weights:   0%|          | 2/760 [00:01<08:51,  1.43it/s]C:\Users\SSAFY\Desktop\AI2_Challenge\downloads\envs\TASK006_baseline_qwen35\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: Fu

| 학습률 | 프롬프트 | 학습 해상도 | 추론 해상도 | 학습 초 | 상태 | Accuracy(%) | 정답 수 | 검증 수 | 1e-4 대비 %p | 개선 | 악화 | 파싱 실패(%) | 초/문항 | 최대 VRAM(GiB) |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 0.0001 | P5 | 640 | 672 | 2729.79 | completed | 94 | 470 | 500 | 0 | 0 | 0 | 0 | 0.865942 | 7.68746 |
| 2e-05 | P5 | 640 | 672 | 2747.53 | completed | 93.6 | 468 | 500 | -0.4 | 4 | 6 | 0 | 0.863416 | 7.68746 |
| 5e-05 | P5 | 640 | 672 | — | not_run | — | — | — | — | — | — | — | — | — |


결과 파일: C:\Users\SSAFY\Desktop\AI2_Challenge\output\TASK-006-lr\LR-6007c4ee02657be3\lr_comparison.csv


C:\Users\SSAFY\Desktop\AI2_Challenge\output\TASK-006-lr\LR-6007c4ee02657be3\lr_comparison.csv

## 6. LR 5e-5

In [12]:
run_stage("train_" + RESOLUTIONS[2])
run_stage(RESOLUTIONS[2])

{'python': '3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]', 'os': 'Windows-10-10.0.26200-SP0', 'torch': '2.11.0+cu128', 'cuda': '12.8', 'gpu': 'NVIDIA GeForce RTX 5060 Ti', 'vram_gib': 15.92828369140625, 'packages': {'transformers': '5.8.0', 'peft': '0.18.1', 'bitsandbytes': '0.49.2', 'accelerate': '1.12.0'}, 'network': 'blocked; local model files only'}
W0922 19:55:47.932000 19016 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d

Loading weights:   0%|          | 2/760 [00:01<08:48,  1.43it/s]C:\Users\SSAFY\Desktop\AI2_Challenge\downloads\envs\TASK006_baseline_qwen35\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: F

| 학습률 | 프롬프트 | 학습 해상도 | 추론 해상도 | 학습 초 | 상태 | Accuracy(%) | 정답 수 | 검증 수 | 1e-4 대비 %p | 개선 | 악화 | 파싱 실패(%) | 초/문항 | 최대 VRAM(GiB) |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 0.0001 | P5 | 640 | 672 | 2729.79 | completed | 94 | 470 | 500 | 0 | 0 | 0 | 0 | 0.865942 | 7.68746 |
| 2e-05 | P5 | 640 | 672 | 2747.53 | completed | 93.6 | 468 | 500 | -0.4 | 4 | 6 | 0 | 0.863416 | 7.68746 |
| 5e-05 | P5 | 640 | 672 | — | not_run | — | — | — | — | — | — | — | — | — |


결과 파일: C:\Users\SSAFY\Desktop\AI2_Challenge\output\TASK-006-lr\LR-6007c4ee02657be3\lr_comparison.csv


C:\Users\SSAFY\Desktop\AI2_Challenge\output\TASK-006-lr\LR-6007c4ee02657be3\lr_comparison.csv

{'python': '3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]', 'os': 'Windows-10-10.0.26200-SP0', 'torch': '2.11.0+cu128', 'cuda': '12.8', 'gpu': 'NVIDIA GeForce RTX 5060 Ti', 'vram_gib': 15.92828369140625, 'packages': {'transformers': '5.8.0', 'peft': '0.18.1', 'bitsandbytes': '0.49.2', 'accelerate': '1.12.0'}, 'network': 'blocked; local model files only'}
W0922 20:41:51.112000 28688 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d

Loading weights:   0%|          | 2/760 [00:01<08:57,  1.41it/s]C:\Users\SSAFY\Desktop\AI2_Challenge\downloads\envs\TASK006_baseline_qwen35\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: F

| 학습률 | 프롬프트 | 학습 해상도 | 추론 해상도 | 학습 초 | 상태 | Accuracy(%) | 정답 수 | 검증 수 | 1e-4 대비 %p | 개선 | 악화 | 파싱 실패(%) | 초/문항 | 최대 VRAM(GiB) |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 0.0001 | P5 | 640 | 672 | 2729.79 | completed | 94 | 470 | 500 | 0 | 0 | 0 | 0 | 0.865942 | 7.68746 |
| 2e-05 | P5 | 640 | 672 | 2747.53 | completed | 93.6 | 468 | 500 | -0.4 | 4 | 6 | 0 | 0.863416 | 7.68746 |
| 5e-05 | P5 | 640 | 672 | 2727.41 | completed | 93.8 | 469 | 500 | -0.2 | 4 | 5 | 0 | 0.862854 | 7.68746 |


결과 파일: C:\Users\SSAFY\Desktop\AI2_Challenge\output\TASK-006-lr\LR-6007c4ee02657be3\lr_comparison.csv


C:\Users\SSAFY\Desktop\AI2_Challenge\output\TASK-006-lr\LR-6007c4ee02657be3\lr_comparison.csv

## 7. 최종 비교표
1e-4 대비 개선/악화 수와 Accuracy, 시간, VRAM을 함께 확인합니다. 자동 채택하지 않습니다.

In [13]:
run_stage("summary")

condition  learning_rate    status   loss_mode prompt  train_resolution  inference_resolution train_status  training_seconds   n  correct_n  accuracy  parse_failure_rate  fallback_usage_rate    seconds  seconds_per_sample  peak_allocated_gib  peak_reserved_gib                                                                                                                        checkpoint  optimizer_updates  training_peak_allocated_gib  reused_baseline  warmup_samples  accuracy_pct  parse_failure_pct  gain  loss  delta_pp
  lr_1e-4        0.00010 completed answer_only     P5               640                   672    completed       2729.790754 500        470     0.940                 0.0                  0.0 432.970805            0.865942            7.687458          11.333984 C:\Users\SSAFY\Desktop\AI2_Challenge\output\TASK-006-lr\LR-6007c4ee02657be3\train_lr_1e-4_20260922_180742_556c7955\adapter_epoch1                250                     10.38922            False               1  

| 학습률 | 프롬프트 | 학습 해상도 | 추론 해상도 | 학습 초 | 상태 | Accuracy(%) | 정답 수 | 검증 수 | 1e-4 대비 %p | 개선 | 악화 | 파싱 실패(%) | 초/문항 | 최대 VRAM(GiB) |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 0.0001 | P5 | 640 | 672 | 2729.79 | completed | 94 | 470 | 500 | 0 | 0 | 0 | 0 | 0.865942 | 7.68746 |
| 2e-05 | P5 | 640 | 672 | 2747.53 | completed | 93.6 | 468 | 500 | -0.4 | 4 | 6 | 0 | 0.863416 | 7.68746 |
| 5e-05 | P5 | 640 | 672 | 2727.41 | completed | 93.8 | 469 | 500 | -0.2 | 4 | 5 | 0 | 0.862854 | 7.68746 |


결과 파일: C:\Users\SSAFY\Desktop\AI2_Challenge\output\TASK-006-lr\LR-6007c4ee02657be3\lr_comparison.csv


C:\Users\SSAFY\Desktop\AI2_Challenge\output\TASK-006-lr\LR-6007c4ee02657be3\lr_comparison.csv

## 결과 확인

실제 결과 경로는 설정 셀 마지막의 `OUTPUT_DIR`에 표시됩니다.

| 파일 | 내용 |
|---|---|
| lr_comparison.csv | LR별 Accuracy·정답 수·파싱 실패·시간·VRAM·1e-4 대비 개선/악화 |
| fixed_train_ids.csv / fixed_valid_ids.csv | 기존 학습·검증 ID와 이미지 그룹 |
| prompt.json | 실제 P5 지시와 프롬프트 예시 |
| train_lr_조건_시각/adapter_epoch1 | 저장 LoRA와 processor |
| train_lr_조건_시각/training_config.json | LR·모델 revision·seed·loss·프롬프트 등 |
| train_lr_조건_시각/supervised_tokens.csv | 처음 3개 학습 문항의 loss 대상 |
| train_lr_조건_시각/train_log.csv | optimizer update별 loss·실제 scheduler LR |
| train_lr_조건_시각/reload_check.json | 저장/재로드 출력 일치 여부 |
| eval_trainlr_조건_시각/valid_predictions.csv | 문항별 출력·정답·실패·이미지 크기/시각 토큰 |
| paired_1e-4_vs_lr_조건.csv / changed_1e-4_vs_lr_조건.csv | 정오 전환 및 변경 문항 |
| type_comparison.csv | 문항 유형별 결과 |
| *.log / requirements.lock.txt / resolution_config.json | 실행 로그·환경·실험 설정 |
| PROJECT_STATUS_update.md / CHANGELOG.md | 00 반영용 요약 및 실제 실행 기록 |

500문항에서 1문제는 0.2%p입니다. 작은 차이는 보류하고 최종 후보에서 반복/추가 검증을 검토하세요.
